(DBCinTSALG)=

# Dirichlet BC in TwoScale method

Symbolic work to verify how to deal with Dirichlet BC when passing from fine scale to coarse scale.

```{attention}
Approach (I) to (IV):
    Works only for shifted enriched function
```
## initialize symbols

In [61]:
from IPython.display import display, Math
import sympy as sp
from sympy import latex as spl_
import numpy as np
sp.init_printing()
def spl(expr):
    return(spl_(expr,mul_symbol='dot'))

The following symbolic variables are representing the different matrices involved in the conputation:

In [62]:

# dummy size to check
dimf=4
dimS=3
dimE=3
dimC=dimS+dimE
dimD=2
dimI=dimS-dimD
dimfD=dimD
dimfd=1
dimfi=dimf-dimfD-dimfd

In [63]:

A=sp.MatrixSymbol('A',dimf,dimf)
B=sp.MatrixSymbol('B',dimf,1)
D=sp.MatrixSymbol('D',dimf,dimf)
U=sp.MatrixSymbol('U',dimf,dimf)
XD=sp.MatrixSymbol('X_D',dimf,1)
PE=sp.MatrixSymbol('P_E',dimf,dimE)
PS=sp.MatrixSymbol('P_S',dimf,dimS)
DC=sp.MatrixSymbol('D_C',dimC,dimC)
UC=sp.MatrixSymbol('U_C',dimC,dimC)
XDC=sp.MatrixSymbol('X_DC',dimC,1)

display(Math(f'{spl(A)} = ~Fine~ matrix'))
display(Math(f'{spl(B)} = ~Fine~ rhs'))
display(Math(f'{spl(PS)} = ~standard~operator~ matrix'))
display(Math(f'{spl(PE)} = ~enriched~operator~ matrix'))
P=sp.BlockMatrix([PS,PE])
display(Math(f'P={spl(P)} = ~full~operator~ matrix'))
display(Math(f'{spl(D)},{spl(U)},{spl(XD)} = ~Dirichlet~ fine~operators'))
display(Math(f'{spl(DC)},{spl(UC)},{spl(XDC)} = ~Dirichlet~ coarse~enriched~operators'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


## Fine scale system


Applying Dirichlet fine operator to $A$ and $B$ gives:

In [64]:
AD=D*A*D+U
BD=-sp.MatMul(D,A,XD)+XD+D*B
display(Math(f' AD = {spl(AD)}'))
display(Math(f' BD = {spl(BD)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>


## TS approach (I)


Applying TS operator to $A$,$B$ gives:

In [65]:
APC=P.transpose()*A*P
BPC=P.transpose()*B
display(Math(f'AP_C={spl(APC)}={spl(sp.block_collapse(APC))}'))
display(Math(f'BP_C={spl(BPC)}={spl(sp.block_collapse(BPC))}'))
APC=sp.block_collapse(APC)
BPC=sp.block_collapse(BPC)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Applying Dirichlet coarse operator to $AP_C$ and $BP_C$ gives:

In [66]:
AC=DC*APC*DC+UC
BC=-DC*APC*XDC+XDC+DC*BPC
display(Math(f'A_C={spl(AC)}'))
display(Math(f'B_C={spl(BC)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>


## TS approach (II)


Applying TS operator to $AD$,$BD$ gives:

In [67]:
APC2=P.transpose()*AD*P
BPC2=P.transpose()*BD
display(Math(f'AP_C^2={spl(APC2)}={spl(sp.block_collapse(APC2))}'))
display(Math(f'BP_C^2={spl(BPC2)}={spl(sp.block_collapse(BPC2))}'))
APC2=sp.block_collapse(APC2)
BPC2=sp.block_collapse(BPC2)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [68]:
AC2=DC*APC2*DC+UC
BC2=-DC*APC2*XDC+XDC+DC*BPC2
display(Math(f'A_C^2={spl(AC2)}'))
display(Math(f'B_C^2={spl(BC2)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Check same matrix

Normally if this approach is correct matrix $A_C^2$ and $A_C$ could be the same. The difference gives

In [69]:

display(Math(f'A_C^2-A_C={spl(AC2-AC)}'))

<IPython.core.display.Math object>

Let's split $DC$ in standard and enriched part as follows:

In [70]:
DCS=sp.MatrixSymbol('D_CS',dimS,dimS)
DCE=sp.MatrixSymbol('D_CE',dimE,dimE)
display(Math(f'{spl(DCS)} = ~standard~block~ of ~D_C'))
display(Math(f'{spl(DCE)} = ~enriched~block~ of ~D_C'))
DCb=sp.BlockDiagMatrix(DCS,DCE)
display(Math(f'D_C={spl(DCb)} = ~rearranged~Dirichlet~operator'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

This leads to:

In [71]:
dAc=sp.block_collapse((AC2-AC).subs([(DC,DCb)]))
display(Math(f'A_C^2-A_C={spl(dAc)}'))


<IPython.core.display.Math object>

As $U=\mathbb{I}-D$ we have:

In [72]:
dAc=sp.block_collapse((AC2-AC).subs([(DC,DCb),(U,sp.Identity(dimf)-D)]))
display(Math(f'A_C^2-A_C={spl(dAc)}'))


<IPython.core.display.Math object>

What can be said about $D.P_S.D_{CS}$:

* $D_{CS}$ is filtering out columns (i.e. set them to zero) of $P_S$  related to eliminated dofs by coarse Dirichlet BC.
* $D$ is filtering out rows (i.e. set them to zero) of $P_S.D_{CS}$  related to eliminated dofs by fine Dirichlet BC.

De facto as imposed Dirichlet boundary conditions at both scale are expected to be the same the following assertion will be true in most cases:

>The eliminated dofs at fine scale are included in the set of row of the eliminated column by $D_{CS}$

This is wrong if for example coarse and fine BC do not stop at same location due to discretisation. In this case we expect to adapt coarse mesh so that it correspond to this stooping location.

If this assertion is true the rows eliminated by $D$ are already null thus:

$D.P_S.D_{CS}=P_S.D_{CS}$   
and  
$U.P_S.D_{CS}=(\mathbb{I}-D).P_S.D_{CS}=P_S.D_{CS}-D.P_S.D_{CS}=P_S.D_{CS}-P_S.D_{CS}=0$

The expression of $A_C^2-A_C$ becomes then:

In [73]:
dAc01=DCS*PS.T*D*A*D*PE*DCE-DCS*PS.T*A*PE*DCE
dAc10=DCE*PE.T*D*A*D*PS*DCS-DCE*PE.T*A*PS*DCS
dAc11=DCE*PE.T*(sp.Identity(dimf)-D+D*A*D)*PE*DCE-DCE*PE.T*A*PE*DCE
dAc=sp.BlockMatrix([[sp.ZeroMatrix(dimS,dimS),dAc01],[dAc10,dAc11]])

display(Math(f'A_C^2-A_C={spl(dAc)}'))



<IPython.core.display.Math object>



But it is clearly not the case for $D.P_E.D_{CE}$ as coarse enriched eliminated dofs if any are not a priori related to equivalent dof of $D_{CS}$ so $D.P_E.D_{CE}\neq P_E.D_{CE}$

But what can be said about $D.A.D-A$ considering that $D=\mathbb{I}-U$

In [74]:


display(Math(f'D.A.D-A={spl((sp.Identity(dimf)-U)*A*(sp.Identity(dimf)-U)-A)}={spl((sp.Identity(dimf)-U)*(A-A*U)-A)}={spl(A-A*U-U*(A-A*U)-A)}={spl(U*A*U -A*U-U*A)}'))



<IPython.core.display.Math object>



The expression of $A_C^2-A_C$ becomes:

In [75]:


dAc01=DCS*PS.T*(U*A*U -A*U-U*A)*PE*DCE
dAc10=DCE*PE.T*(U*A*U -A*U-U*A)*PS*DCS
dAc11=DCE*PE.T*(U+U*A*U -A*U-U*A)*PE*DCE
dAc=sp.BlockMatrix([[sp.ZeroMatrix(dimS,dimS),dAc01],[dAc10,dAc11]])
display(Math(f'A_C^2-A_C={spl(dAc)}'))

<IPython.core.display.Math object>



And as already mentioned $U.P_S.D_{CS}=0$ thus expression simplify further:

In [76]:


dAc01=DCS*PS.T*(-A*U)*PE*DCE
dAc10=DCE*PE.T*(-U*A)*PS*DCS
dAc11=DCE*PE.T*(U+U*A*U -A*U-U*A)*PE*DCE
dAc=sp.BlockMatrix([[sp.ZeroMatrix(dimS,dimS),dAc01],[dAc10,dAc11]])
display(Math(f'A_C^2-A_C={spl(dAc)}'))

<IPython.core.display.Math object>



Which is not null as there is no reason that $U.P_E$ is null. So it cannot further be simplified except if we consider that enriched dof are not eliminated and in this case $D_{CE}=\mathbb{I}$ and $A_C^2-A_C$ becomes:

In [77]:


display(Math(f'A_C^2-A_C={spl(sp.block_collapse(dAc.subs([(DCE,sp.Identity(dimE))])))}'))

<IPython.core.display.Math object>



In conclusion **$A_C^2$ and $A_C$ are not the same**. 

### check same system

Matrix are not the same but maybe systems are giving same solutions. From last expression of $dA_C^2=A_C^2-A_C$ (the one with $D_{CE}$ not forcefully $\mathbb{I}$ we can write:

In [78]:
display(Math(f'X_C={spl(AC.I*BC)}'))

<IPython.core.display.Math object>


And if both system give the same solution $A_C^2.X_C$ should be equal to $B_C^2$:

$A_C^2.X_C=(A_C+dA_C^2).X_C=B_C+dA_C^2.A_C^{-1}.B_C=B_C^2$

or
$B_C^2-B_C=dA_C^2.A_C^{-1}.B_C$

we have, splitting $X_{DC}$ in its two contributions $X_{DCS}$ $X_{DCE}$:

In [79]:
XDCS=sp.MatrixSymbol('X_DCS',dimS,1)
XDCE=sp.MatrixSymbol('X_DCE',dimE,1)
XDCb=sp.BlockMatrix([[XDCS],[XDCE]])
display(Math(f'{spl(XDC)}={spl(XDCb)}'))
dBc0=sp.block_collapse((BC2-BC).subs([(DC,DCb),(XDC,XDCb)]))
display(Math(f'B_C^2-B_C={spl(dBc0)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

using
$D.P_S.D_{CS}=P_S.D_{CS}$ and $U.P_S.D_{CS}=0$

$B_C^2-B_C$ is

In [80]:
dBc1=sp.block_collapse(sp.expand(dBc0).subs([(DCS*PS.T*D,DCS*PS.T)]))
dBc2=sp.block_collapse(sp.expand(dBc1).subs([(U*PS*DCS,sp.ZeroMatrix(dimf,dimS)),(DCS*PS.T*U,sp.ZeroMatrix(dimS,dimf))]))
display(Math(f'B_C^2-B_C={spl(dBc2)}'))

<IPython.core.display.Math object>

But what is $X_{DCE}$ ? It is the vector of potential imposed value applied to some enriched dofs. If any enriched dofs are imposed they will be used to eliminated rank deficiency in problem and will thus certainly be set to zero. So we can expect that if $X_{DCE}$ exist it will be a null vector. So we will set this hypothesis:

> $X_{DCE}=0$

With this hypothesis expression becomes:

In [81]:
dBc=sp.block_collapse(sp.expand(dBc2).subs([(XDCE,sp.ZeroMatrix(dimE,1))]))
display(Math(f'B_C^2-B_C={spl(dBc)}'))

<IPython.core.display.Math object>

On the other hand we have $dA_C^2.A_C^{-1}.B_C$ which is not easy to manipulate. But if we could have expressed $B_C^2-B_C$ as $H.B_C$ then relation to pouve would be:

$H.B_C=dA_C^2.A_C^{-1}.B_C$

$(H-dA_C^2.A_C^{-1}).B_C=0$

Then 
* if $B_C=0$ then $H$ do not exist
* if $B_C\neq0$ either $H-dA_C^2.A_C^{-1}$ is orthogonal to $B_C$  or $H-dA_C^2.A_C^{-1}=0$

In this last case

$H=dA_C^2.A_C^{-1}$

$H.A_C=dA_C^2$

not much simple to handle.

**It is hard to conclude**


## TS  approch (III)


This approach is a simple reorganization of the approach (I) to optimize implementation. First we rewrite final system using block $D_C$

In [82]:

AC30=sp.block_collapse(AC.subs([(DC,DCb)]))
BC30=sp.block_collapse(BC.subs([(DC,DCb)]))
display(Math(f'A_C^3={spl(AC30)}'))
display(Math(f'B_C^3={spl(BC30)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>



Then we set :

In [83]:

QS=sp.MatrixSymbol('Q_S',dimf,dimS)
QE=sp.MatrixSymbol('Q_E',dimf,dimE)
display(Math(f'{spl(QS)}={spl(PS*DCS)}'))
display(Math(f'{spl(QE)}={spl(PE*DCE)}'))



<IPython.core.display.Math object>

<IPython.core.display.Math object>



System becomes

In [84]:

AC31=sp.block_collapse(AC30.subs([(PS*DCS,QS),(PE*DCE,QE),(DCS*PS.T,QS.T),(DCE*PE.T,QE.T)]))
BC31=sp.block_collapse(BC30.subs([(PS*DCS,QS),(PE*DCE,QE),(DCS*PS.T,QS.T),(DCE*PE.T,QE.T),(XDC,XDCb)]))
display(Math(f'A_C^3={spl(AC31)}'))
display(Math(f'B_C^3={spl(BC31)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>



We see  that for $A_C^3$ construction we don't need to keep $P_S$,$P_E$ during TS loop. Only $Q_S$,$Q_E$ are required. But for $B_C^3$ we need $P_E$,$P_S$ and $A$ !!!

But one can observe that $P_S.X_{DCS}$ is  constant during TS loop and can be set as a $W_S$ vector.
Thus

In [85]:
WS=sp.MatrixSymbol('W_S',dimf,1)
ZS=sp.MatrixSymbol('Z_S',dimf,1)


display(Math(f'Z_S=A.P_S.{spl(XDCS)}={spl(A*WS)}'))


<IPython.core.display.Math object>



Is also a constant during TS loop. Thus we can write $B_C^3$ as:

In [86]:


BC32=sp.simplify(sp.block_collapse(BC31.subs([(A*PS*XDCS,ZS)])))
display(Math(f'B_C^3={spl(BC32)}'))

<IPython.core.display.Math object>



Now concerning $A.P_E.X_{DCE}$ if we look only at this formal expression it has to be update at all TS iteration as $P_E$ change.    
But if we follow the same hypotheses as above ($X_{DCE}=0$) the final system is:

In [87]:

BC33=sp.block_collapse(BC32.subs([(XDCE,sp.ZeroMatrix(dimE,1))]))
display(Math(f'A_C^3={spl(AC31)}'))
display(Math(f'B_C^3={spl(BC33)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>



From an implementation point of view it is perfect as $P_E$,$P_S$ are not needed anymore and applying  $Q_S$,$Q_E$ is doing the job of applying the TS operator and the coarse Dirichlet BC at the same time.

But in the TS Loop we project the solution $X_C$ of the problem at coarse scale on fine scale field using $P_E$,$P_S$ as follows :

In [88]:
XCS=sp.MatrixSymbol('X_CS',dimS,1)
XCE=sp.MatrixSymbol('X_CE',dimE,1)
XC=sp.BlockMatrix([[XCS],[XCE]])

STS=sp.block_collapse(P*(AC31.I*BC33))

display(Math(f'STS={spl(STS)}={spl(P*XC)}'))

<IPython.core.display.Math object>



Here we are going to use the fact that $P.\mathbb{I}=P.(D_C+U_C)=Q+P.U_{C}$ so the complement of $Q$ for $P$ is:


In [89]:
LS=sp.MatrixSymbol('L_S',dimf,dimS)
LE=sp.MatrixSymbol('L_E',dimf,dimE)
L=sp.BlockMatrix([LS,LE])

Ls=sp.block_collapse(P*UC)

display(Math(f'L=P-Q={spl(L)}={spl(Ls)}'))

<IPython.core.display.Math object>



So  the projected solution is:

In [90]:
Q=sp.BlockMatrix([QS,QE])

display(Math(f'STS={spl(P*XC)}={spl((L+Q)*XC)}={spl(sp.block_collapse((Ls+Q)*XC))}'))

<IPython.core.display.Math object>



And as  $X_C$ filtered by $U_C$ gives in fact $X_{DC}$ as they are related to the same dofs, we have $L.X_C=P.U_C.X_C=P.X_{DC}=[W_S+P_E.X_{DCE}]$  
and as we consider above that $X_{DCE}=0$  
$L.X_C=[W_S+0]$


In [91]:


display(Math(f'STS={spl(sp.block_collapse(WS+Q*XC))}'))

<IPython.core.display.Math object>



From an implementation point of view it is again perfect as $P_E$,$P_S$ are not needed anymore and only  $Q_S$,$Q_E$ and $W_S$ are required


## TS  approach (IV)


This approach is  motivated by the use of Petsc nested matrix format. In this case sub block are treated independently. The context is to consider that no enriched dof are eliminated and to leverage matrix manipulation by really eliminating Dirichlet boundary condition from the system. Also enriched space is now really limited to only enriched dof of enriched nodes (i.e. no Dirichlet). This approach is thus a reorganization of the approach (I) like approach (III). First we add the following definition:

 * The standard set is split in Dirichlet imposed dof and free dofs called hereafter reduced dofs
 * The enriched set is only having dofs related to enriched nodes (no Dirichlet) also called reduced dofs hereafter
 * $R$ an operator to passe from reduced set to full set

In [92]:


XRD=sp.MatrixSymbol('X_RD',dimD,1)
XRS=sp.MatrixSymbol('X_RS',dimI,1)
XRE=sp.MatrixSymbol('X_RE',dimE,1)
#XR=sp.BlockMatrix([[XRD],[XRS],[XRE]])
XR=sp.BlockMatrix([[XRS],[XRE]])

G=sp.MatrixSymbol('G',dimS-dimI,dimE)


display(Math(f'{spl(XRD)} = ~Imposed~Dirichlet~dof~of~standard~set'))
display(Math(f'{spl(XRS)} = ~Reduced~standard~dof'))
display(Math(f'{spl(XRE)} = ~Reduced~enriched~dof = {spl(XCE)}'))
display(Math(f'Reduced~coarse~vector ={spl(XR)}'))

R=sp.BlockMatrix([[sp.ZeroMatrix(dimD,dimI),sp.ZeroMatrix(dimD,dimE)],[sp.Identity(dimI),sp.ZeroMatrix(dimI,dimE)],[sp.ZeroMatrix(dimE,dimI),sp.Identity(dimE)]])
display(Math(f'R={spl(R)} = ~full~restriction~operator~ matrix'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


And similarly to approach (III) we gone mix boundary condition and operator. This is done here by splitting $P_S$ in its Dirichlet column block $P_{SD}$ and its reduced column block $P_{SR}$. For enriched operator we use again the name $P_E$ but now column are in this approach restricted only to enriched nodes (no Dirichlet):

In [93]:

PSD=sp.MatrixSymbol('P_SD',dimf,dimD)
PSR=sp.MatrixSymbol('P_SR',dimf,dimI)
PR=sp.BlockMatrix([PSD,PSR,PE])


display(Math(f'{spl(PSD)} = ~Standard~operator~restricted~to~Dirichlet~dofs'))
display(Math(f'{spl(PSD)} = ~Standard~operator~restricted~to~reduced~dofs'))
display(Math(f'Full~TS~operator ={spl(PR)}'))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>



The solution at coarse level is:

In [94]:
XCD=sp.BlockMatrix([[XRD],[sp.ZeroMatrix(dimI,1)],[sp.ZeroMatrix(dimE,1)]])
XC=sp.block_collapse(R*XR)+XCD

display(Math(f'X_C={spl(R)}.{spl(XR)}+{spl(XCD)}={spl(XC)}={spl(sp.block_collapse(XC))}'))


<IPython.core.display.Math object>

Using first step of approach (I) we have:

In [95]:
PAPC=PR.transpose()*A*PR
PBC=PR.transpose()*B
display(Math(f'PAP_C={spl(PAPC)}={spl(sp.block_collapse(PAPC))}'))
display(Math(f'BP_C={spl(PBC)}={spl(sp.block_collapse(PBC))}'))
PAPC=sp.block_collapse(PAPC)
PBC=sp.block_collapse(PBC)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

And now applying the reduction operator:

In [96]:
AC4=R.transpose()*PAPC*R
BC4=R.transpose()*(PBC-PAPC*XCD)

display(Math(f'A_C^4={spl(AC4)}={spl(sp.block_collapse(AC4))}'))
display(Math(f'B_C^4={spl(BC4)}={spl(sp.block_collapse(BC4))}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Like in approach (III) we can set

In [97]:
W_=sp.MatrixSymbol('W',dimf,1)
Z_=sp.MatrixSymbol('Z',dimf,1)
W=PSD*XRD
Z=A*W
display(Math(f'W={spl(W)}'))
display(Math(f'Z={spl(Z)}={spl(A)}*{spl(W_)}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

which are constant during TS loop and can be computed once at initialization of the loop. The system is then expressed as:

In [98]:
BC4=R.transpose()*PBC-sp.BlockMatrix([PSR,PE]).transpose()*Z_

display(Math(f'A_C^4={spl(sp.block_collapse(AC4))}'))
display(Math(f'B_C^4={spl(sp.block_collapse(BC4))}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

And regarding projection on fine computation we have:

In [99]:

STS=sp.block_collapse(PR*(R*AC4.I*BC4+XCD))

display(Math(f'STS={spl(STS)}={spl(PR*(R*XR+XCD))}'))
display(Math(f'STS={spl(sp.block_collapse(PR*(R*XR+XCD)))}'))
display(Math(f'STS={spl(W_)}+{spl(PSR*XRS)}+{spl(PE*XRE)}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


So like in approach (III) it is possible to mix boundary condition and operator application and get the following minimal computations :
* create  $P_E$
* create $P_{SR}$ and temporary $P_{SD}$
* compute $W$
* remove $P_{SD}$
* compute once at the initialization of the loop:
    * $Z$
    * $P_{SR}^t.(B-Z)$ block of the VecNest vector $B_C^4$
    *  $P_{SR}^t.A.P_{SR}$ block of the MatNest matrix $A_C^4$ 
* In the loop at each TS iteration:
    * compute $P_E$
    * compute each block of the MatNest matrix $A_C^4$ related to $P_E$
    * compute $P_{E}^t.(B-Z)$ block of the VecNest vector $B_C^4$
    * Solve the system
    * compute $STS$ 



## TS  approch (V)


This approach is just the generalization of approach (IV) to general enriched function. Imposing Dirichlet is now done on the combination of the standard and enriched dof as enriched function is not anymore null at enriched nodes. 
This correspond to the addition of cinematic equation of the form:

$X_{D}-G_E.X_{RE}=X_{RD}$

where
*  $X_{D}$ are the standard dof involved in a cinematic equation (i.e. dof on enriched node where we impose Dirichlet BC)
*  $G_E$ is and operator constructed from enriched function values:
    * rows correspond to the set of dof related to $X_D$ (standard dof eliminated)
    * column correspond to the set of enriched dof related to $X_D$
*  $X_{RD}$ remain the same: the imposed Dirichlet values

```{note}
These extra equations exist only if the Dirichlet boundary condition is applied on an enriched node. Otherwise it is a usual BC elimination. This can be imposed by setting a null row in $G_E$ for these non enriched nodes. If all Dirichlet boundary condition are imposed on non enriched nodes $G_E$ is null and consequently there is no difference with approach (IV).
```

This extra equations can be added naturally to the restriction operator $R$ by eliminating $X_{D}$ which gives

In [100]:


G=sp.MatrixSymbol('G_E',dimD,dimE)

R=sp.BlockMatrix([[sp.ZeroMatrix(dimD,dimI),G],[sp.Identity(dimI),sp.ZeroMatrix(dimI,dimE)],[sp.ZeroMatrix(dimE,dimI),sp.Identity(dimE)]])
display(Math(f'R={spl(R)} = ~full~restriction~operator~matrix~with~cinematic~relations'))


<IPython.core.display.Math object>

The solution at corse level is given by:

In [101]:
#XCD=sp.BlockMatrix([[XRD],[sp.ZeroMatrix(dimI,1)],[sp.ZeroMatrix(dimE,1)]])
XC=sp.block_collapse(R*XR)+XCD

display(Math(f'X_C={spl(R)}.{spl(XR)}+{spl(XCD)}={spl(XC)}={spl(sp.block_collapse(XC))}'))


<IPython.core.display.Math object>

where we recognize the expression of $X_D$ on the first line.

The system and the projection are:

In [102]:
AC5=R.transpose()*PAPC*R
BC5=R.transpose()*(PBC-PAPC*XCD)

display(Math(f'A_C^5={spl(AC5)}={spl(sp.block_collapse(AC5))}'))
display(Math(f'B_C^5={spl(BC5)}={spl(sp.block_collapse(BC5))}'))

AC5=sp.block_collapse(AC5)
BC5=sp.block_collapse(BC5)

STS=sp.block_collapse(PR*(R*XR+XCD))

display(Math(f'STS={spl(PR*(R*XR+XCD))}={spl(STS)}'))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Let add the following:

In [103]:
W_=sp.MatrixSymbol('W',dimf,1)
Z_=sp.MatrixSymbol('Z',dimf,1)
TE_=sp.MatrixSymbol('T_E',dimf,dimE)


W=PSD*XRD
TE=PE+PSD*G
display(Math(f'W={spl(W)}'))
display(Math(f'Z={spl(Z)}={spl(A)}{spl(W_)}'))
display(Math(f'T_E={spl(TE)}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Then the system and the projection are:

In [104]:
AC5_=sp.block_collapse(sp.expand(AC5.subs([(PE.transpose()*A,(TE_.transpose()-G.transpose()*PSD.transpose())*A)])))
BC5_=BC5.subs([(A*PSD*XRD,Z_),(PE.transpose(),TE_.transpose()-G.transpose()*PSD.transpose())])
STS_=sp.expand(sp.expand(STS).subs([(PE,TE_-PSD*G),(PSD*XRD,W_)]))
AC5_=sp.block_collapse(sp.expand(AC5_.subs([(A*PE,A*TE_-A*PSD*G)])))
BC5_=sp.block_collapse(sp.expand(BC5_))

display(Math(f'A_C^5={spl(AC5_)}'))
display(Math(f'B_C^5={spl(BC5_)}'))
display(Math(f'STS={spl(STS_)}'))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

From a formal point of view this approach is the same as approach (IV) with $T_E$ taking the role of $P_E$.

Let look at $T_E$.It is $P_E$ plus $P_{SD}G_E$. $P_{SD}$  are the column of $P_S$ related to Dirichlet dofs. Multiplied by $G_E$ it modify column of $P_E$ related to enriched dofs on node where Dirichlet BC are applied. And more precisely this operation correspond in fact to use shifted enriched function for those particular dof/column of $P_E$


Thus the computational path is almost the same as the one used in approach (IV):
* create  $P_E$ and identify column of $P_E$ related to node with Dirichlet BC
* create $P_{SR}$ and temporary $P_{SD}$
* compute $W$
* remove $P_{SD}$
* compute once at the initialization of the loop:
    * $Z$
    * $P_{SR}^t.(B-Z)$ block of the VecNest vector $B_C^5$
    *  $P_{SR}^t.A.P_{SR}$ block of the MatNest matrix $A_C^5$ 
* In the loop at each TS iteration:
    * compute $T_E$ (like $P_E$ but with shifted enriched function for Diriclet column)
    * compute each block of the MatNest matrix $A_C^5$ related to $T_E$
    * compute $T_{E}^t.(B-Z)$ block of the VecNest vector $B_C^5$
    * Solve the system
    * compute $STS$ 



## TS  approch (VI)

In approach (IV) and (V) elimination of Dirichlet boundary condition by restriction operator lead to obtain system with sizes that are not anymore a multiple of a block size (bs=nb components per node). To keep this property, like in conventional Dirichlet treatment in FEniCSx/PETSc, using a null row/column operator plus associated identity addition can do the job. This reactivate approach (II) as now we can imagine to use fine system with eliminate Dirichlet to generate blocks related to standard dof.

Lets start by renaming everything considering general enriched function and potential imposed Dirichlet boundary condition to enriched nodes. This last point add the following equations:

$X_{SD}-G_E.X_{E}=C_{SD}$

or 

$X_{SD}=G_E.X_{E}+C_{SD}$


with:




In [105]:
XS_=sp.MatrixSymbol('X_S',dimS,1)
XSD=sp.MatrixSymbol('X_SD',dimD,1)
CSD=sp.MatrixSymbol('C_SD',dimD,1)
XSI=sp.MatrixSymbol('X_SI',dimI,1)
XE=sp.MatrixSymbol('X_E',dimE,1)
XS=sp.BlockMatrix([[XSD],[XSI]])
XC=sp.BlockMatrix([[XSD],[XSI],[XE]])
XC_=sp.MatrixSymbol('X_C',dimC,1)
XI_=sp.MatrixSymbol('X_I',dimI+dimE,1)
XI=sp.BlockMatrix([[XSI],[XE]])
XH_=sp.MatrixSymbol('X_H',dimC,1)
XH=sp.BlockMatrix([[CSD],[sp.ZeroMatrix(dimI,1)],[sp.ZeroMatrix(dimE,1)]])

display(Math(rf"{spl(XSD)}:\text{{Standard constrained/dependent dofs }}(nb_D\times 1)"))
display(Math(rf"{spl(XSI)}:\text{{Standard independent dofs }}(nb_I\times 1)"))
display(Math(rf"{spl(XS_)}={spl(XS)}:\text{{All Standard dofs }}(nb_{{std}}\times 1)"))
display(Math(rf"{spl(XE)}:\text{{Enriched dofs }}(nb_{{enr}}\times 1)"))
display(Math(rf"{spl(XC_)}={spl(XC)}:\text{{All  coarse dofs }}((nb_{{std}}+nb_{{enr}})\times 1)"))
display(Math(rf"{spl(XI_)}={spl(XI)}:\text{{All  coarse independent dofs }}((nb_{{I}}+nb_{{enr}})\times 1)"))
display(Math(rf"{spl(XH_)}={spl(XH)}:\text{{Constrain vector with imposed values from equations }}((nb_{{std}}+nb_{{enr}})\times 1)"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

So conventionally we set a $T$ cinematic operator as follows:


In [106]:

T=sp.BlockMatrix([[sp.ZeroMatrix(dimD,dimI),G],[sp.Identity(dimI),sp.ZeroMatrix(dimI,dimE)],[sp.ZeroMatrix(dimE,dimI),sp.Identity(dimE)]])

display(Math(rf"T={spl(T)} : \text{{full cinematic elimination operator matrix}}"))


<IPython.core.display.Math object>

And the coarse solution $X_C$ vector can be expressed as:

In [107]:
XCcalc=sp.block_collapse(T*XI+XH)

display(Math(f'{spl(XC_)}={spl(XC)}={spl(T)}.{spl(XI)}+{spl(XH)}={spl(XCcalc)}'))


<IPython.core.display.Math object>

where we recognize the equation expression of $X_{SD}$ on the first line.

Conventionally the system 

$A_{CC}.X_C=B_C$

becomes

$A_{CC}.(T.X_I+X_H)=B_C$

or

$A_{CC}.T.X_I=B_C-A_{CC}.X_H$

And by pre multiplying by $T^t$ to re obtain a square matrix:

$T^t.A_{CC}.T.X_I=T^t.(B_C-A_{CC}.X_H)$

But as mentioned above in this approach we don't want to have a reduced size system. Thus we embedded this system into a larger one by simply add dummy equation corresponding to constrained set with the addition of dummy unknowns $X_{Dumy}$.

$T$ becomes



In [108]:

TZ=sp.BlockMatrix([[sp.ZeroMatrix(dimD,dimD),sp.ZeroMatrix(dimD,dimI),G],[sp.ZeroMatrix(dimI,dimD),sp.Identity(dimI),sp.ZeroMatrix(dimI,dimE)],[sp.ZeroMatrix(dimE,dimD),sp.ZeroMatrix(dimE,dimI),sp.Identity(dimE)]])

display(Math(rf"T={spl(TZ)} : \text{{full cinematic elimination operator matrix with dimension kept }}"))


<IPython.core.display.Math object>

And the solution is then given by:

In [109]:
XdumySD_=sp.MatrixSymbol('X_SDdumy',dimD,1)
XCdumy_=sp.MatrixSymbol('X_CSDdumy',dimC,1)
XCdumy=sp.BlockMatrix([[XdumySD_],[XSI],[XE]])
XCcalc=sp.block_collapse(TZ*XCdumy+XH)


display(Math(f'{spl(XC_)}={spl(XC)}=T.{spl(XCdumy_)}+X_H={spl(TZ)}.{spl(XCdumy_)}+{spl(XH)}={spl(XCcalc)}'))


<IPython.core.display.Math object>

The sytem is then
$A_{CC}.(T.X_{CSDdumy}+X_H)=B_C$

or

$A_{CC}.T.X_{CSDdumy}=B_C-A_{CC}.X_H$

And by pre multiplying by $T^t$ to re obtain a square matrix:

$T^t.A_{CC}.T.X_{CSDdumy}=T^t.(B_C-A_{CC}.X_H)$

But now the sub block $SD\times SD$ is null so we need to add 1 to the diagonal of this sub block to make it invertible. We call $U_C$ like in previous approach the matrix containing this modified sub block :



In [110]:

UC=sp.BlockMatrix([[sp.Identity(dimD),sp.ZeroMatrix(dimD,dimI),sp.ZeroMatrix(dimD,dimE)],[sp.ZeroMatrix(dimI,dimD),sp.ZeroMatrix(dimI,dimI),sp.ZeroMatrix(dimI,dimE)],[sp.ZeroMatrix(dimE,dimD),sp.ZeroMatrix(dimE,dimI),sp.ZeroMatrix(dimE,dimE)]])

display(Math(rf"U_C={spl(UC)} : \text{{Identity matrix for sub block }} SD\times SD"))


<IPython.core.display.Math object>

The final form of the solved system is then:

$(T^t.A_{CC}.T+U_C).X_{CSDdumy}=T^t.(B_C-A_{CC}.X_H)$

And the coarse solution is :

$U_C= T.(T^t.A_{CC}.T+U_C)^{-1}.T^t.(B_C-A_{CC}.X_H)+X_H$

When $G_E$ is null (with shift enrichment function it can be the case) then these two steps can be grouped in one with the solution of the system being directly the searched coarse solution. $T$ is then simply $D_C$ defined in previous approach:

$(D_C^t.A_{CC}.D_C+U_C).X_{C}=D_C^t.(B_C-A_{CC}.X_H)+X_H$

Let's now split TS operator considering constrain at coarse level and Dirichlet at fine level. 

We have




In [111]:
PS_DD_=sp.MatrixSymbol('PS_DD',dimfD,dimD)
PS_dD_=sp.MatrixSymbol('PS_dD',dimfd,dimD)
PS_iD_=sp.MatrixSymbol('PS_iD',dimfi,dimD)
PS_DI_=sp.MatrixSymbol('PS_DI',dimfD,dimI)
PS_dI_=sp.MatrixSymbol('PS_dI',dimfd,dimI)
PS_iI_=sp.MatrixSymbol('PS_iI',dimfi,dimI)
PE_DE_=sp.MatrixSymbol('PE_DE',dimfD,dimE)
PE_dE_=sp.MatrixSymbol('PE_dE',dimfd,dimE)
PE_iE_=sp.MatrixSymbol('PE_iE',dimfi,dimE)

P_=sp.MatrixSymbol('P',dimf,dimC)

P=sp.BlockMatrix([[PS_DD_,PS_DI_,PE_DE_],[PS_dD_,PS_dI_,PE_dE_],[PS_iD_,PS_iI_,PE_iE_]])

display(Math(rf"{spl(P_)}={spl(P)}:\text{{TS operator split in blocks }}(nb_{{f}}\times {{nb_{{c}}}})"))

<IPython.core.display.Math object>


Note that imposed constraint $C_{SD}$ correspond exactly to Dirichlet imposed value at fine scale on common coarse location. And thus by construction
$PS_{DI}$ is null and $PS_{DD}$ is the identity as only $D$ set coarse dof are related by a coefficient of 1 with the $D$ set fine dofs.
Here we also consider that fine Dirichlet boundary condition are embedded in coarse Dirichlet BC so d set dof are forcefully related only to D set coarse dof and then $PS_{dI}$ is null. So $P$ simplifies to:



In [112]:



P=sp.BlockMatrix([[sp.Identity(dimfD),sp.ZeroMatrix(dimfD,dimI),PE_DE_],[PS_dD_,sp.ZeroMatrix(dimfd,dimI),PE_dE_],[PS_iD_,PS_iI_,PE_iE_]])

display(Math(rf"{spl(P_)}={spl(P)}"))

<IPython.core.display.Math object>


To simplify for now we group TS operator in standard and enriched part:

In [113]:

PS_=sp.MatrixSymbol('PS',dimf,dimS)
PE_=sp.MatrixSymbol('PE',dimf,dimE)


PS=sp.BlockMatrix([[sp.Identity(dimfD),sp.ZeroMatrix(dimfD,dimI)],[PS_dD_,sp.ZeroMatrix(dimfd,dimI)],[PS_iD_,PS_iI_]])
PE=sp.BlockMatrix([[PE_DE_],[PE_dE_],[PE_iE_]])
P__=sp.BlockMatrix([PS_,PE_])
display(Math(rf"{spl(P_)}={spl(P)}={spl(P__)}"),Math(r"\text{{with}}"))
display(Math(rf"{spl(PS_)}={spl(PS)}"))
display(Math(rf"{spl(PE_)}={spl(PE)}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


And split the same way the fine system:

In [114]:
A_DD_=sp.MatrixSymbol('A_DD',dimfD,dimfD)
A_Dd_=sp.MatrixSymbol('A_Dd',dimfD,dimfd)
A_Di_=sp.MatrixSymbol('A_Di',dimfD,dimfi)
A_dD_=sp.MatrixSymbol('A_dD',dimfd,dimfD)
A_dd_=sp.MatrixSymbol('A_dd',dimfd,dimfd)
A_di_=sp.MatrixSymbol('A_di',dimfd,dimfi)
A_iD_=sp.MatrixSymbol('A_iD',dimfi,dimfD)
A_id_=sp.MatrixSymbol('A_id',dimfi,dimfd)
A_ii_=sp.MatrixSymbol('A_ii',dimfi,dimfi)


A_=sp.MatrixSymbol('A',dimf,dimf)

A=sp.BlockMatrix([[A_DD_,A_Dd_,A_Di_],[A_dD_,A_dd_,A_di_],[A_iD_,A_id_,A_ii_]])

display(Math(rf"{spl(A_)}={spl(A)}:\text{{Unconstrained fine scale system matrix }}(nb_{{f}}\times {{nb_{{f}}}})"))

B_D_=sp.MatrixSymbol('B_D',dimfD,1)
B_d_=sp.MatrixSymbol('B_d',dimfd,1)
B_i_=sp.MatrixSymbol('B_i',dimfi,1)

B_=sp.MatrixSymbol('B',dimf,1)

B=sp.BlockMatrix([[B_D_],[B_d_],[B_i_]])

display(Math(rf"{spl(B_)}={spl(B)}:\text{{Unconstrained fine scale system rhs }}(nb_{{f}}\times 1"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

The Dirichlet boundary condition operators at fine scale are:

In [115]:

D=sp.BlockMatrix([[sp.ZeroMatrix(dimfD,dimfD),sp.ZeroMatrix(dimfD,dimfd),sp.ZeroMatrix(dimfD,dimfi)],[sp.ZeroMatrix(dimfd,dimfD),sp.ZeroMatrix(dimfd,dimfd),sp.ZeroMatrix(dimfd,dimfi)],[sp.ZeroMatrix(dimfi,dimfD),sp.ZeroMatrix(dimfi,dimfd),sp.Identity(dimfi)]])
U=sp.BlockMatrix([[sp.Identity(dimfD),sp.ZeroMatrix(dimfD,dimfd),sp.ZeroMatrix(dimfD,dimfi)],[sp.ZeroMatrix(dimfd,dimfD),sp.Identity(dimfd),sp.ZeroMatrix(dimfd,dimfi)],[sp.ZeroMatrix(dimfi,dimfD),sp.ZeroMatrix(dimfi,dimfd),sp.ZeroMatrix(dimfi,dimfi)]])
Cd=sp.MatrixSymbol('C_d',dimfd,1)
Xh=sp.BlockMatrix([[CSD],[Cd],[sp.ZeroMatrix(dimfi,1)]])

display(Math(rf"D={spl(D)}:\text{{Dirichlet elimination operator }}(nb_f\times nb_f)"))
display(Math(rf"U={spl(U)}:\text{{Dirichlet complementary operator }}(nb_f\times nb_f)"))
display(Math(rf"X_h={spl(Xh)}:\text{{Dirichlet imposed values (for D set these are the same that what is imposed at coase scale) }}(nb_f\times 1)"))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Thus the fine system with eliminated Dirichlet is:


In [116]:

AD=D*A*D+U
BD=D*(B-A*Xh)+Xh
AD_=sp.MatrixSymbol('AD',dimf,dimf)
BD_=sp.MatrixSymbol('BD',dimf,1)

display(Math(rf"{spl(AD_)}={spl(AD)}={spl(sp.block_collapse(AD))}:\text{{constrained fine scale system matrix  }}(nb_f\times nb_f)"))
display(Math(rf"{spl(BD_)}={spl(BD)}={spl(sp.block_collapse(BD))}:\text{{constrained fine scale system rhs  }}(nb_f\times 1)"))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

Lets now apply the TS operator to both system:

In [117]:
ATS6=P.transpose()*A_*P
BTS6=P__.transpose()*B_
display(Math(rf'P^t.A.P={spl(ATS6)}={spl((sp.block_collapse(P.transpose()*A*P)))}'))
display(Math(rf'P^t.B={spl(BTS6)}={spl(sp.block_collapse(P.transpose()*B))}'))

ADTS6=P__.transpose()*AD_*P__
BDTS6=P__.transpose()*BD_
display(Math(rf'P^t.AD.P={spl(ADTS6)}={spl((sp.block_collapse(P.transpose()*AD*P)))}'))
display(Math(rf'P^t.BD={spl(BDTS6)}={spl(sp.block_collapse(P.transpose()*BD))}'))

#STS=sp.block_collapse(PR*(R*XR+XCD))

#display(Math(f'STS={spl(PR*(R*XR+XCD))}={spl(STS)}'))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

and apply the coarse constrains $T$ to form the system 

In [118]:
AC6=TZ.transpose()*P.transpose()*A*P*TZ+UC
ADC6=TZ.transpose()*P.transpose()*AD*P*TZ+UC

display(Math(rf'A_C^6={spl(AC6)}={spl((sp.block_collapse(AC6)))}'))
display(Math(rf'AD_C^6={spl(ADC6)}={spl((sp.block_collapse(ADC6)))}'))
AC6=sp.block_collapse(AC6)
ADC6=sp.block_collapse(ADC6)

BC6=TZ.transpose()*(P.transpose()*B-P.transpose()*A*P*XH)+XH
BDC6=TZ.transpose()*(P.transpose()*BD-P.transpose()*AD*P*XH)+XH


display(Math(rf'B_C^6={spl(BC6)}={spl(sp.block_collapse(BC6))}'))
display(Math(rf'BD_C^6={spl(BDC6)}={spl(sp.block_collapse(BDC6))}'))
BC6=sp.block_collapse(BC6)
BDC6=sp.block_collapse(BDC6)


display(Math(rf'dA={spl((sp.block_collapse(sp.expand(ADC6-AC6))))}'))
display(Math(rf'dB={spl((sp.block_collapse(sp.expand(BDC6-BC6))))}'))


#BC6=sp.block_collapse(BC6)

#STS=sp.block_collapse(PR*(R*XR+XCD))

#display(Math(f'STS={spl(PR*(R*XR+XCD))}={spl(STS)}'))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

dA is not null as seen in approach (II) but the (SD+SI)x(SD+SI) block is.

dB is not null but SDx1 is. And if we further impose that d set fine boundary condition is a linear interpolation of coarse D then $C_d=PS_{dD}.C_{SD}$. Then it is in this case the(SD+SI)x1 which is null.

The question is then: can we mix use of $A$,$B$ and $AD$,$BD$ per block to get directly $A_C^6$ correctly without adding $U_C$ which impose extra matrix operation with PETSc.

Not clear for now ......

Let us $Q$,$W$,$Z$ like in previous approaches:

In [132]:
Q=P*TZ
display(Math(rf"Q=P.T={spl(Q)}={spl(sp.block_collapse(Q))}"))
Q=sp.block_collapse(Q)
Q_=sp.MatrixSymbol('Q',dimf,dimC)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

And lets imagine the modified $Q$:

In [139]:
Q2=Q+sp.BlockMatrix([[sp.Identity(dimD),sp.ZeroMatrix(dimD,dimI),sp.ZeroMatrix(dimD,dimE)],[sp.ZeroMatrix(dimfd,dimD),sp.ZeroMatrix(dimfd,dimI),sp.ZeroMatrix(dimfd,dimE)],[sp.ZeroMatrix(dimfi,dimD),sp.ZeroMatrix(dimfi,dimI),sp.ZeroMatrix(dimfi,dimE)]])
display(Math(rf"Qb={spl(Q2)}={spl(sp.block_collapse(Q2))}"))
Q2=sp.block_collapse(Q2)


<IPython.core.display.Math object>

Then we can obtain a corect sub block (stdxstd) with this operator:

In [142]:
AC62=Q2.transpose()*A*Q2
ADC62=Q2.transpose()*AD*Q2
display(Math(rf'A_C^6={spl(AC62)}={spl((sp.block_collapse(AC62)))}'))
display(Math(rf'AD_C^6={spl(ADC62)}={spl((sp.block_collapse(ADC62)))}'))
display(Math(rf'ADb-A={spl((sp.block_collapse(sp.expand(ADC62-AC6))))}'))
display(Math(rf'Ab-A={spl((sp.block_collapse(sp.expand(AC62-AC6))))}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

But it cost the price of having $Q$ and $Qb$ in memory, which is too much, because $Qb$ is ok for sub block (stdxstd) but not for the other sub block.

It is cheaper to use $U_C$ even if adding it will cost some reallocation but as it is done only once at the beginning of the loop it can be ok.

Conlusion this **approch is not giving anything**.